# A — Đồ thị tiên quyết cho các KC của MathDial (Achieve the Core Coherence Map)

**Đóng ý kiến #4** (R-b Q10, Stanford RPKT): MathDial không có đồ thị tiên quyết, nên cơ chế cốt lõi
của MTA (Advance theo tiên quyết) chưa bao giờ được kích hoạt trên dữ liệu thật.

**Phương pháp (cập nhật 15/09/2026): dùng đồ thị do chuyên gia soạn thay cho LLM.**
KC của MathDial trong bản chú thích LLMKT (`tag_src="atc"`) là **nguyên văn mô tả chuẩn Common Core theo
Achieve the Core**, nên khớp được trực tiếp với mã chuẩn và với liên kết *progress from* (tiên quyết) của
Achieve the Core Coherence Map — không cần suy luận của mô hình, không cần GPU.

**Vì sao bỏ phương án Llama-3.1-8B**:
lần chạy trên L4 gán 84/103 KC vào "lớp 6", nên bộ lọc ứng viên sai từ gốc; 220 cạnh Llama chỉ trùng 29 cạnh
Coherence Map, 6 cạnh ngược chiều, và chỉ tìm lại 16/145 cạnh chuyên gia.

**Bốn bước** (cài đặt trong `cpu/09_build_atc_graph.py`, notebook gọi thẳng script để không có hai bản logic):
1. **Khớp KC → mã chuẩn**: trùng khớp sau chuẩn hoá (91 KC); nếu không, KC là tiền tố của mô tả ATC — ATC thêm
   chú thích "Grade N expectations … limited to …" (12 KC). Kết quả 103/103, mỗi KC một mã riêng.
2. **Gộp chuẩn con** (sub-standard) vào KC cha; lớp và domain lấy từ mã chuẩn (K = 0, trung học = 9).
3. **Dựng cạnh**: đi ngược *progress from*; gặp KC khác → thêm cạnh; gặp chuẩn nằm ngoài 103 KC → đi xuyên qua
   (co lập đồ thị qua node trung gian, để không mất quan hệ A → X → B).
4. **Tự kiểm và rút gọn**: dừng nếu có chu trình hoặc cạnh từ lớp cao về lớp thấp; rút gọn bắc cầu
   (bỏ A → C khi đã có A → … → C).

**Đầu ra** (`revision_kit/out/`): `prereq_graph_mathdial_atc.json` → nạp vào `cpu/08_endtoend_mta.py`;
`prereq_edges_atc.csv`; và `kc_ccss_map.csv`. Đồ thị dựng hoàn toàn bằng quy tắc, không chỉnh tay.

**Nguồn và trích dẫn bắt buộc** (ODC-BY 1.0): dataset `allenai/achieve-the-core` (arXiv:2408.04226) và
Achieve the Core Coherence Map. Chi tiết: `out/external/NGUON.md`.

In [ ]:
import os, json, hashlib
import pandas as pd

try:                                                                  # Colab: doc tu Drive
    from google.colab import drive; drive.mount("/content/drive")
    KIT = "/content/drive/MyDrive/multi-agents-knowledge-tracing"         # <<< ban clone repo nay tren Drive
    if not os.path.exists(KIT):                                          # lan dau: tu clone tu GitHub
        import subprocess; subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/manhhdv/multi-agents-knowledge-tracing.git", KIT], check=True)
except ImportError:                                                   # may thuong: tim thu muc goc repo tu thu muc hien tai
    here = os.getcwd()
    KIT = next((p for p in [here] + [os.path.abspath(os.path.join(here, *[".."] * i)) for i in (1, 2, 3)]
                if os.path.exists(os.path.join(p, "cpu", "09_build_atc_graph.py"))), None)
SCRIPT = os.path.join(KIT or "", "cpu", "09_build_atc_graph.py")
assert os.path.exists(SCRIPT), "Khong tim thay thu muc goc repo (can cpu/09_build_atc_graph.py)"
os.environ["MATHKT_ROOT"] = KIT
OUT = os.path.join(KIT, "out")

# Du lieu Achieve the Core (kem san trong repo): dung dung phien ban da kiem
STD = os.path.join(OUT, "external", "achieve-the-core_standards.jsonl")
STD_SHA256 = "01d6235cca86185f01ab2486b132eb554db44543a07905f6b2adb3bf34755d77"
assert os.path.exists(STD), "Thieu out/external/achieve-the-core_standards.jsonl"
sha = hashlib.sha256(open(STD, "rb").read()).hexdigest()
assert sha == STD_SHA256, f"standards.jsonl khac phien ban da dung ({sha[:12]}...) - kiem lai truoc khi dung"
print("repo:", KIT); print("Achieve the Core:", os.path.getsize(STD), "byte, sha256 OK")

## Bước 1–4 — khớp chuẩn, dựng cạnh, tự kiểm, rút gọn bắc cầu

Chạy `cpu/09_build_atc_graph.py`; script tự dừng nếu có KC không khớp duy nhất, chu trình, hoặc cạnh ngược lớp.

In [ ]:
get_ipython().run_line_magic("run", f'-i "{SCRIPT}"')

### Kiểm tra nhanh kết quả

In [ ]:
M = pd.read_csv(os.path.join(OUT, "kc_ccss_map.csv"))
E = pd.read_csv(os.path.join(OUT, "prereq_edges_atc.csv"))
print("So KC theo lop (0 = K, 9 = trung hoc):"); display(M.groupby("grade").size().rename("so_KC").to_frame().T)
print("So canh theo khoang cach lop:"); display(E.grade_gap.value_counts().sort_index().rename("so_canh").to_frame().T)
print("KC la tien quyet cua nhieu KC nhat:"); display(E.prereq_id.value_counts().head(8).rename("so_KC_dich").to_frame().T)
display(E[["prereq_id", "kc_id", "prereq", "kc"]].head(10))